# Multi-Agent Workflow — `@azure/ai-projects`

This notebook demonstrates how to create a multi-agent workflow with a student agent answering the question first and then a teacher agent checking the answer, processing streaming responses with workflow action events.

It mirrors the [`workflowMultiAgent.ts`](./workflowMultiAgent.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

## Prerequisites

1. **Build the package first** so `dist/` is current: `cd sdk/ai/ai-projects && pnpm build`
2. **tslab kernel** installed and registered (`npm install -g tslab` then `tslab install`); select the **TypeScript** (tslab) kernel.
3. **Launch VS Code / Jupyter from `sdk/ai/ai-projects/`** so Node resolves the local `@azure/ai-projects`.
4. **`az login`** completed so `DefaultAzureCredential` can authenticate.
5. **Environment variables**: `FOUNDRY_PROJECT_ENDPOINT`, `FOUNDRY_MODEL_NAME`.

Run the cells in order (top to bottom); state is shared across cells.

In [2]:
// Imports and configuration
import { DefaultAzureCredential } from "@azure/identity";
import { AIProjectClient } from "@azure/ai-projects";

const projectEndpoint = process.env["FOUNDRY_PROJECT_ENDPOINT"] ?? "<project endpoint>";
const deploymentName = process.env["FOUNDRY_MODEL_NAME"] ?? "<model deployment name>";
console.log(`Model: ${deploymentName}`);

Model: gpt-5.2


In [3]:
// Create the AI Project client and the OpenAI client
const project = new AIProjectClient(projectEndpoint, new DefaultAzureCredential());
// Annotated as `any` so tslab does not try to emit a non-portable declaration
// referencing the deep `node_modules/openai` (pnpm junction) path.
const openAIClient: any = project.getOpenAIClient();

In [4]:
// Create Teacher Agent
console.log("Creating teacher agent...");
const teacherAgent = await project.agents.createVersion("teacher-agent", {
  kind: "prompt",
  model: deploymentName,
  instructions: `You are a teacher that create pre-school math question for student and check answer.
                 If the answer is correct, you stop the conversation by saying [COMPLETE].
                 If the answer is wrong, you ask student to fix it.`,
});
console.log(
  `Agent created (id: ${teacherAgent.id}, name: ${teacherAgent.name}, version: ${teacherAgent.version})`,
);

Creating teacher agent...
Agent created (id: teacher-agent:67, name: teacher-agent, version: 67)


In [5]:
// Create Student Agent
console.log("\nCreating student agent...");
const studentAgent = await project.agents.createVersion("student-agent", {
  kind: "prompt",
  model: deploymentName,
  instructions: `You are a student who answers questions from the teacher.
                 When the teacher gives you a question, you answer it.`,
});
console.log(
  `Agent created (id: ${studentAgent.id}, name: ${studentAgent.name}, version: ${studentAgent.version})`,
);


Creating student agent...
Agent created (id: student-agent:68, name: student-agent, version: 68)


In [6]:
// Create Multi-Agent Workflow
console.log("\nCreating multi-agent workflow...");
const workflowYaml = `
kind: workflow
trigger:
  kind: OnConversationStart
  id: my_workflow
  actions:
    - kind: SetVariable
      id: set_variable_input_task
      variable: Local.LatestMessage
      value: "=UserMessage(System.LastMessageText)"

    - kind: CreateConversation
      id: create_student_conversation
      conversationId: Local.StudentConversationId

    - kind: CreateConversation
      id: create_teacher_conversation
      conversationId: Local.TeacherConversationId

    - kind: InvokeAzureAgent
      id: student_agent
      description: The student node
      conversationId: "=Local.StudentConversationId"
      agent:
        name: ${studentAgent.name}
      input:
        messages: "=Local.LatestMessage"
      output:
        messages: Local.LatestMessage

    - kind: InvokeAzureAgent
      id: teacher_agent
      description: The teacher node
      conversationId: "=Local.TeacherConversationId"
      agent:
        name: ${teacherAgent.name}
      input:
        messages: "=Local.LatestMessage"
      output:
        messages: Local.LatestMessage

    - kind: SetVariable
      id: set_variable_turncount
      variable: Local.TurnCount
      value: "=Local.TurnCount + 1"

    - kind: ConditionGroup
      id: completion_check
      conditions:
        - condition: '=!IsBlank(Find("[COMPLETE]", Upper(Last(Local.LatestMessage).Text)))'
          id: check_done
          actions:
            - kind: EndConversation
              id: end_workflow

        - condition: "=Local.TurnCount >= 4"
          id: check_turn_count_exceeded
          actions:
            - kind: SendActivity
              id: send_activity_tired
              activity: "Let's try again later...I am tired."

      elseActions:
        - kind: GotoAction
          id: goto_student_agent
          actionId: student_agent
`;

const workflow = await project.agents.createVersion(
  "student-teacher-workflow",
  {
    kind: "workflow",
    workflow: workflowYaml,
  },
  {
    foundryFeatures: "WorkflowAgents=V1Preview",
  },
);
console.log(
  `Agent created (id: ${workflow.id}, name: ${workflow.name}, version: ${workflow.version})`,
);


Creating multi-agent workflow...
Agent created (id: student-teacher-workflow:17, name: student-teacher-workflow, version: 17)


In [7]:
// Create conversation
console.log("\nCreating conversation...");
const conversation = await openAIClient.conversations.create();
console.log(`Created conversation (id: ${conversation.id})`);


Creating conversation...
Created conversation (id: conv_f4a261003ef3cb2100G3h3WTmncpLMffJRqV0rBu265kTPVfiQ)


In [8]:
// Send request to the workflow with streaming
console.log("\nSending request to multi-agent workflow with streaming...");
const stream = await openAIClient.responses.create(
  {
    conversation: conversation.id,
    input: "1 + 1 = ?",
    stream: true,
  },
  {
    body: {
      agent_reference: { name: workflow.name, type: "agent_reference" },
      metadata: { "x-ms-debug-mode-enabled": "1" },
    },
  },
);

// Process the streaming response
const iterator = stream[Symbol.asyncIterator]();
let next = await iterator.next();
while (!next.done) {
  const event = next.value;
  console.log("Event received:", JSON.stringify(event, null, 2));
  if (event.type === "response.output_item.added" || event.type === "response.output_item.done") {
    const item = event.item;
    console.log(`\n ${JSON.stringify(item, null, 2)} added:`);
  }
  next = await iterator.next();
}


Sending request to multi-agent workflow with streaming...
Event received: {
  "type": "response.created",
  "sequence_number": 1,
  "response": {
    "metadata": {
      "x-ms-debug-mode-enabled": "1"
    },
    "service_tier": "auto",
    "model": "",
    "background": false,
    "tools": [],
    "truncation": "auto",
    "id": "wfresp_f4a261003ef3cb2100bNgO6u6VnM5UM5F5QT3zXaAQn4hCWcUr",
    "object": "response",
    "status": "in_progress",
    "created_at": 1784853158,
    "completed_at": 1784853158,
    "error": null,
    "incomplete_details": null,
    "output": [],
    "instructions": "",
    "parallel_tool_calls": true,
    "conversation": {
      "id": "conv_f4a261003ef3cb2100G3h3WTmncpLMffJRqV0rBu265kTPVfiQ"
    },
    "agent_reference": {
      "type": "agent_reference",
      "name": "student-teacher-workflow",
      "version": "17"
    }
  }
}
Event received: {
  "type": "response.in_progress",
  "sequence_number": 2,
  "response": {
    "metadata": {
      "x-ms-debug-mod

In [9]:
// Clean up
console.log("\nCleaning up resources...");
await openAIClient.conversations.delete(conversation.id);
console.log("Conversation deleted");

await project.agents.deleteVersion(workflow.name, workflow.version);
console.log("Workflow deleted");

await project.agents.deleteVersion(studentAgent.name, studentAgent.version);
console.log("Student Agent deleted");

await project.agents.deleteVersion(teacherAgent.name, teacherAgent.version);
console.log("Teacher Agent deleted");

console.log("\nMulti-agent workflow sample completed!");


Cleaning up resources...
Conversation deleted
Workflow deleted
Student Agent deleted
Teacher Agent deleted

Multi-agent workflow sample completed!
